# Data Modeling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/data-modeling.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Data modeling is where most analytics problems are actually decided. A warehouse can be fast and still give the wrong revenue number if the grain, keys, or history handling are wrong. The questions below cover the Kimball vocabulary that interviewers expect (facts, dimensions, grain, SCD, bridges, conformed dimensions), the SQL and PySpark that implements it, and the trade-offs that decide between a star schema, a snowflake, and a Data Vault.

## What is data modeling in analytics and why does it matter?

Data modeling in analytics is the process of structuring data so it can be queried reliably and efficiently for business questions. A good model makes metrics consistent, reduces complexity for consumers, and improves performance by aligning storage with query patterns. A bad model leads to inconsistent KPIs, confusing joins, and expensive queries.

The usual layered shape of a modern warehouse is:

```text
sources (OLTP, APIs, events, files)
        |
        v
staging / bronze      raw, append-only, one table per source
        |
        v
integration / silver  cleaned, conformed, deduplicated entities
        |
        v
marts / gold          facts + dimensions at a defined grain, documented metrics
        |
        v
BI, notebooks, reverse ETL, feature stores
```

Three decisions recur at every layer:

1. **Grain** - what does one row represent?
2. **Keys** - natural, surrogate, or composite, and how they join.
3. **Change handling** - overwrite (SCD1), version history (SCD2), or append-only events.

Kimball-style dimensional modeling puts facts and dimensions in the gold layer; Inmon-style modeling normalizes the integration layer first. Most practical warehouses borrow from both: normalized staging, dimensional marts.

Why it matters in interviews: a candidate who starts with "what is the grain and what does the business call a customer?" is showing modeling discipline; a candidate who starts writing `JOIN`s is showing SQL knowledge but not design judgment.

Mental model: an OLTP database is modeled for correct writes, and an analytics model is modeled for correct reads. OLTP normalizes to 3NF so each fact is stored once and an update cannot create a contradiction: one row per current order, customer, product. OLAP keeps the same business rules but reshapes data around questions: a fact table at a declared grain plus dimensions that describe it. The trade is write complexity for read simplicity.

How it works in practice:
+ Normalization removes redundancy by splitting entities and relationships. 1NF makes columns atomic, 2NF removes partial key dependencies, 3NF removes transitive dependencies. It protects writes but forces many joins per report.
+ Dimensional modeling keeps wide, denormalized dimensions plus a narrow fact so a report needs one join per dimension and no more.
+ Additivity follows from the grain: quantity is additive, a balance is semi-additive (you cannot sum it across days), and a ratio is non-additive (compute it from numerator and denominator).

Worked example: an orders application stores `orders` (one row per order, with shipping and tax) and `order_lines` (one row per line). The analytics model keeps the same entities but changes the grain contract: `fact_order_line` is one row per line, and order-level amounts are either allocated down to lines with a documented rule or kept in a separate `fact_order` at order grain. Nothing is lost, and every measure has exactly one home.

Common mistake: copying source tables 1:1 into the warehouse and calling it a model. A 1:1 copy inherits the source grain, so every consumer re-derives the same metric and gets a different answer. Model around the question, not the source schema.

## What is grain and why is it the first question to answer?

Grain is the level of detail of a table, for example "one row per order line" or "one row per user per day". It determines which metrics are additive, how joins behave, and what can be uniquely identified. If grain is unclear, you will get duplicates, incorrect aggregations, and ambiguous keys.

Write the grain as an English sentence before writing any DDL, then encode it as a uniqueness test:

```sql
-- grain: one row per order line
select order_id, line_number, count(*) as c
from fact_order_line
group by order_id, line_number
having count(*) > 1;
```

```sql
-- grain: one row per account per day
select account_id, snapshot_date, count(*) as c
from fact_account_balance_daily
group by account_id, snapshot_date
having count(*) > 1;
```

Common grain mistakes:

+ Storing order-level values (shipping, tax) on every order-line row, then summing them.
+ Mixing snapshot grain with event grain in one table so a `sum` silently multiplies.
+ Using a table with no declared primary key, so nothing prevents duplicate loads.

Interview tip: grain and the primary key are the same statement expressed twice. If the key and the grain disagree, the model is wrong.

Grain is a contract with three parts: the entity, the period or event, and the uniqueness rule. "One row per order line" means `(order_id, line_number)` is unique. "One row per account per day" means `(account_id, snapshot_date)` is unique. Write all three parts before naming a column.

Grain also fixes how every measure behaves:
+ Additive across all dimensions: quantity, amount.
+ Semi-additive across some dimensions, usually time: account balance, inventory on hand. Use `avg`, `min`, `max`, or last value over time, never `sum`.
+ Non-additive: ratios and percentages. Store the numerator and denominator and compute the ratio at query time.

Gotcha: a table can look correct and still mix two grains. If `group by order_id, line_number` returns one row per line but `shipping_amount` repeats the order-level value, the key is unique and the grain is still wrong.

Worked example: the requirement is "revenue by product by month, and average order value". The lowest grain that answers both is one row per order line, because average order value is `sum(line_amount) / count(distinct order_id)`. Monthly product totals alone could never produce it.

Common mistake: choosing the grain of today's dashboard instead of the process. If the report later moves from monthly to weekly, a monthly fact cannot be split back apart. Prefer the lowest grain you can afford to store; the exception is an event stream so large that a coarser periodic snapshot is the deliberate design.

## What is the difference between fact and dimension tables?

Facts store measurable events (orders, payments, clicks) and typically contain foreign keys to dimensions. Dimensions store descriptive attributes (user, product, date) that provide context for slicing and filtering. Facts answer "what happened", dimensions answer "who/what/where/when".

```sql
create table dim_customer (
  customer_sk   bigint      primary key,
  customer_id   varchar(50) not null,
  customer_name varchar(200),
  segment       varchar(50),
  country_code  char(2)
);

create table fact_order_line (
  order_line_sk  bigint primary key,
  order_id       varchar(50) not null,   -- degenerate dimension
  customer_sk    bigint references dim_customer (customer_sk),
  product_sk     bigint references dim_product (product_sk),
  order_date_sk  int    references dim_date (date_sk),
  quantity       int,
  gross_amount   numeric(14, 2),
  discount_amount numeric(14, 2)
);
```

Practical rules:

+ Facts are usually narrow, long, and partitioned by date; dimensions are wide, short, and replicated.
+ Numeric columns in a fact table should be additive or semi-additive measures, not codes.
+ Descriptive text belongs in dimensions, not in the fact, unless it is a degenerate dimension such as `order_id`.

Think of a fact table as the answer to "what happened, how much, and when", and dimensions as the vocabulary you slice it by. Every fact row is an event or a snapshot at the declared grain; every dimension row is a durable entity described by many attributes.

Mechanically:
+ The fact holds numeric measures plus foreign keys to dimensions. It is tall and narrow, and it grows with events.
+ Dimensions hold descriptive text and flags. They are short and wide, and they grow with entities, not events.
+ A dimension key in the fact must be the dimension surrogate key, so the join is many-to-one and no fact row is duplicated.

Gotcha: not every numeric column is a measure. Discount percentage, tax rate, and latitude are numeric but descriptive and belong in a dimension. Conversely `order_id` is an identifier with no attributes, so it lives in the fact as a degenerate dimension rather than its own table.

Worked example: `fact_order_line` stores `quantity`, `gross_amount`, `discount_amount` plus `customer_sk`, `product_sk`, `order_date_sk`. Everything you filter by without arithmetic, such as name, category, segment, or country, is in a dimension. Adding `product_category` to the fact for speed is a denormalization decision, not a modeling one, and it still needs one owner.

Common mistake: putting descriptive text in the fact to avoid a join. The text repeats once per event, so a rename becomes a mass update and the fact carries bytes that never change.

## What is a star schema and what are its benefits?

A star schema has a central fact table connected to multiple dimension tables. Benefits include simpler queries, clear join paths, and good performance because joins are typically fact-to-dimension (many-to-one). It also encourages consistent definitions of metrics and attributes.

```text
        dim_date
           |
dim_product -- fact_order_line -- dim_customer
           |
        dim_store
```

```sql
select d.year_month,
       p.category,
       sum(f.gross_amount - f.discount_amount) as net_revenue,
       sum(f.quantity)                          as units
from fact_order_line f
join dim_date    d on d.date_sk = f.order_date_sk
join dim_product p on p.product_sk = f.product_sk
where d.year_month = '2026-10'
group by d.year_month, p.category;
```

Why BI tools like stars: every dimension joins to the fact on a single surrogate key, so the tool can infer the join graph, generate `GROUP BY` correctly, and avoid fan-out. Snowflaked dimensions force the tool (and the analyst) through several hops, which increases the chance of a wrong aggregate.

Mental model: a star is a fact surrounded by flat dimensions. The shape matters because every dimension join is many-to-one, so the fact row count never changes and every measure stays additive.

How it works:
+ Each dimension has a single-column surrogate primary key; the fact references it.
+ Dimensions are denormalized on purpose: one wide row per member with every useful attribute.
+ The star extends with a bridge table for many-to-many relationships and a separate fact for header-level measures, without breaking the join paths.

Gotcha: adding a second fact to the same diagram does not make it a star. A report that joins two facts at different grains fans out. Aggregate one fact to a shared grain first, or route the relationship through a bridge.

```sql
-- full star DDL with surrogate keys and a declared grain
create table dim_product (
  product_sk   bigint primary key,
  product_id   varchar(50) not null,
  product_name varchar(200),
  category     varchar(100),
  department   varchar(100),
  is_current   boolean
);

create table fact_order_line (
  order_line_sk bigint primary key,
  order_id      varchar(50) not null,   -- degenerate dimension
  line_number   int not null,
  product_sk    bigint not null references dim_product (product_sk),
  order_date_sk int    not null,
  quantity      int    not null,
  net_amount    numeric(14, 2) not null,
  unique (order_id, line_number)        -- grain guard
);
```

Common mistake: a dimension that is not unique on its key, for example `dim_customer` with one row per address. Every fact row duplicates at join time and revenue doubles. The star contract is one row per dimension key; keep addresses or SCD2 history in their own tables and join to the current or as-of row.

## Star schema vs snowflake schema: what are the trade-offs?

Snowflake schema normalizes dimensions into multiple related tables. It can reduce redundancy and improve dimension maintainability, but it increases join complexity and can hurt performance. Star schemas are usually preferred for BI and ad-hoc analytics because they are easier for users and tools.

Snowflaked product dimension:

```sql
create table dim_product (
  product_sk  bigint primary key,
  product_id  varchar(50),
  product_name varchar(200),
  category_sk bigint references dim_category (category_sk),
  brand_sk    bigint references dim_brand (brand_sk)
);

create table dim_category (
  category_sk bigint primary key,
  category_name varchar(100),
  department_sk bigint references dim_department (department_sk)
);
```

The same information flattened for a star:

```sql
create table dim_product (
  product_sk     bigint primary key,
  product_id     varchar(50),
  product_name   varchar(200),
  category_name  varchar(100),
  department_name varchar(100),
  brand_name     varchar(100)
);
```

| Aspect | Star | Snowflake |
|---|---|---|
| Join count for a report | 1 hop per dimension | multiple hops |
| Storage | more redundancy | less redundancy |
| Ease for BI users | high | lower |
| Dimension maintenance | rewrite a wide row | update a small table |
| Query performance | usually better | usually worse |
| Best fit | BI marts, self-service | very large, shared dimensions with many consumers |

Interview answer: use a star for the consumption layer. Snowflake the dimension only when a sub-attribute is genuinely shared by many dimensions (for example a global geography or cost-center hierarchy) or when the dimension is so large that redundant text is a real cost. Many teams keep a snowflake in the warehouse and publish flattened views or semantic-layer datasets on top.

Normalization and star schemas optimize different costs. Normalizing removes redundancy, so storage and update cost fall, but each report joins more tables and the optimizer has more work. A star denormalizes the same attributes into one table, so storage rises while the join count per report stays at one per dimension.

Where the trade tips on an MPP or columnar engine:
+ Repeated text compresses well, so the storage saving from snowflaking is often small.
+ Small dimensions are broadcast to every node, so a one-hop star join is cheap; a chain of dimension joins may force an extra shuffle.
+ A sub-attribute shared by many dimensions, such as a global geography or cost-center hierarchy, is the strongest case for snowflaking, because one update must apply everywhere.

Worked example: a product hierarchy of department, category, and subcategory shared by 40 million fact rows. Snowflaked, a report joins `fact` then `dim_product` then `dim_category` then `dim_department`. Flattened, all three text columns sit on `dim_product` and the report joins once. Both return the same numbers; the star scans a wider dimension but joins once.

Common mistake: snowflaking for storage and then adding flattened views for every consumer, so the team maintains two shapes that drift. Pick one shape per layer and publish the other as a generated view, never a hand-maintained table.

## What are the main types of fact tables?

Common types are:

+ transactional: one row per business event (order, payment)
+ periodic snapshot: one row per entity per period (daily account balance)
+ accumulating snapshot: one row that is updated as a process progresses (order lifecycle)

```sql
-- transactional
create table fact_payment (
  payment_sk bigint primary key,
  customer_sk bigint,
  payment_date_sk int,
  amount numeric(14, 2)
);

-- periodic snapshot: one row per account per day
create table fact_account_balance_daily (
  account_sk bigint,
  snapshot_date_sk int,
  closing_balance numeric(18, 2),
  primary key (account_sk, snapshot_date_sk)
);

-- accumulating snapshot: one row per order, milestones updated in place
create table fact_order_fulfillment (
  order_sk bigint primary key,
  ordered_date_sk int,
  packed_date_sk int,
  shipped_date_sk int,
  delivered_date_sk int,
  days_to_ship int
);
```

Choosing the right type depends on the questions you need to answer. Activity questions ("how many?") want a transactional fact; balance and state questions ("how many were open?") want a periodic snapshot; process questions ("how long did each stage take?") want an accumulating snapshot.

Four fact patterns cover most models:
+ Transaction fact: one row per event, insert-only, sparse because measures exist only for the covered process.
+ Periodic snapshot: one row per entity per period, dense even when nothing changed, and usually semi-additive.
+ Accumulating snapshot: one row per process instance, updated in place as milestones complete, with lag columns between milestone dates.
+ Factless fact: one row per relationship or event with no measure, used for coverage and eligibility.

Gotcha: transaction and periodic snapshot answer different questions and cannot substitute for each other. "How many payments today" is a transaction fact; "what was the balance on the 15th" is a periodic snapshot. Storing a daily balance on every transaction row makes the sum meaningless.

Worked example: an order process needs both. `fact_order_line` at transaction grain records what was sold. `fact_order_fulfillment` at accumulating-snapshot grain holds one row per order with `ordered_date_sk`, `packed_date_sk`, `shipped_date_sk`, `delivered_date_sk`, and lag columns. Fulfillment metrics come from date differences; sales metrics come from the transaction fact.

Common mistake: building a transaction fact for a state question such as open tickets, then netting deltas by hand. Model the state as a periodic snapshot and the metric becomes a count of open rows.

## What is a surrogate key and when should you use it?

A surrogate key is a synthetic identifier used instead of a natural key. It is useful when natural keys are not stable, when you need SCD history, or when integrating multiple sources with different key systems. Surrogate keys also allow consistent joins even if source keys change.

```sql
-- deterministic surrogate key built from the natural key(s)
select
  abs(xxhash64(cast(customer_id as varchar))) as customer_sk,
  customer_id,
  customer_name
from stg_customer;
```

In [ ]:
# map a natural key to a stable integer surrogate key in PySpark
from pyspark.sql import functions as F

customers = (
    customers
    .withColumn("customer_sk", F.abs(F.xxhash64("source_system", "customer_id")))
)

Design notes:

+ Use a hash (deterministic, parallel-safe) or an identity/sequence (compact, but requires a lookup). Hash keys are easier in distributed loads because there is no coordination.
+ A type-2 dimension needs a *new* surrogate key per version; the natural key stays the same across versions.
+ Keep surrogate keys meaningless - never encode business meaning into digits of the key.
+ Do not use natural keys as the physical primary key of a large fact table: they are wide, and source systems sometimes reuse or recycle them.

Mental model: a natural key identifies a thing in the source; a surrogate key identifies a row in your dimension. Surrogate keys exist so the warehouse controls identity and versioning.

How it works:
+ Type 1 (overwrite) keeps one surrogate key per entity forever.
+ Type 2 (history) issues a new surrogate key per version, so the natural key repeats.
+ Facts store the surrogate key resolved at load time, which is what makes point-in-time joins exact.
+ A deterministic hash such as `xxhash64(business_key)` is parallel-safe because every worker computes the same value. A sequence or identity needs a central generator or a reserved key range.

Gotcha: do not build a surrogate key from the attributes you track for SCD2. If the hash includes `segment`, a segment change mints a new key for the same entity and cross-version reporting breaks. Hash the business key only, and add a version number or `is_current` for identity.

Worked example: two source systems both use `customer_id`. The hub key is `xxhash64(source_system || customer_id)`, so the same digits in two systems never collide. The version key is the hub key plus `valid_from`, which makes each SCD2 row unique without a central sequence.

Common mistake: reusing a source key as the fact foreign key and then finding that a customer rename changes historical reports. A rename is an attribute change; the surrogate key must stay stable for that entity version.

## What is a conformed dimension?

A conformed dimension is a dimension shared consistently across multiple fact tables or data marts. For example, the same "customer" dimension used for orders and support tickets. Conformed dimensions enable consistent reporting and cross-domain analysis.

```text
dim_customer
   |             |
fact_order    fact_support_ticket
```

Rules that keep a dimension conformed:

+ One owner and one definition per attribute. "Customer" cannot mean "billing account" in one mart and "user" in another.
+ Identical surrogate keys and column names wherever the dimension is published.
+ Conformed dimensions are what make drill-across (`order revenue` vs `ticket count` by customer segment) trustworthy.
+ Date is the most common conformed dimension and should always be shared.

A dimension is conformed when two facts use the same physical table, the same keys, and the same attribute definitions. That sameness is what makes drill-across, such as comparing order revenue and support tickets by customer segment, valid.

How to keep it conformed:
+ One owner, one refresh schedule, and one definition per attribute.
+ The same surrogate key space wherever the dimension is joined.
+ A bus matrix records which facts capture which conformed dimensions.

Gotcha: two marts that each build their own `dim_customer` with different segment logic are not conformed even when the column names match. The drift stays silent until a cross-domain report disagrees, and then history has to be rebuilt.

Worked example: `dim_date` and `dim_customer` are conformed across `fact_order_line`, `fact_payment`, and `fact_support_ticket`. A drill-across groups all three by `dim_customer.segment` at each fact's own grain, then joins the small aggregate results on segment. It never joins fact to fact directly.

Common mistake: conforming the keys but not the attributes. Identical `customer_sk` values with one mart treating a closed account as active and another as inactive produce a segment report that cannot be reconciled.

## What is SCD (Slowly Changing Dimension)?

SCD describes how you handle changes in dimension attributes over time (for example a user changing address). Type 1 overwrites history, Type 2 tracks history with multiple rows and effective dates, and Type 3 keeps limited history in additional columns. The choice depends on whether "as-of" reporting is required.

| Type | Behavior | Keeps history | Typical use |
|---|---|---|---|
| 0 | never changes | attribute frozen | immutable codes |
| 1 | overwrite in place | no | fixing typos, corrections |
| 2 | new row + effective dates | full | segment, address, plan, ownership |
| 3 | previous value in an extra column | one prior value | simple "before/after" reporting |
| 4 | separate history table | full, outside main dim | audit-heavy, huge dimensions |
| 6 | 2 + 3 hybrid | full + current column on every row | power-user reporting |

Interview guidance: pick the type per attribute, not per table. A `customer_name` correction is SCD1; a `customer_segment` change that affects historical revenue attribution is SCD2.

Type 6 is the hybrid worth naming. It carries the full Type 2 history columns (`valid_from`, `valid_to`, `is_current`, `version`) plus a current-value column that holds the latest value on every historical row. A report can filter `is_current` for the current picture and still join history by date.

Column mechanics to know from memory:
+ Type 1: one row, attribute overwritten, no dates.
+ Type 2: `valid_from`, `valid_to` (or `effective_start_at`, `effective_end_at`), `is_current`, `version`, new surrogate key per version.
+ Type 3: current value plus `previous_value` and a changed-at timestamp.
+ Type 6: Type 2 columns plus the current value backfilled onto all versions.

Gotcha: `valid_to` must be exclusive at the next version's `valid_from`, and exactly one row per key may have `is_current = true`. Overlapping ranges make a point-in-time join return two rows and double the measure.

Worked example for customer `C123`, first a segment change, then an email correction applied as Type 1 on the open row:

| customer_sk | customer_id | segment | email | valid_from | valid_to | is_current | version |
|---|---|---|---|---|---|---|---|
| 1001 | C123 | standard | a@x.com | 2026-01-01 | 2026-07-01 | false | 1 |
| 1002 | C123 | enterprise | a@x.com | 2026-07-01 | 9999-12-31 | true | 2 |

The email correction updates row 1002 in place and creates no version, because email is not tracked for history.

Common mistake: using `NULL` for the open end date. `NULL` breaks comparisons unless every query remembers `coalesce`, so use a sentinel such as `9999-12-31` consistently.

## How do you implement SCD Type 2?

SCD2 typically stores:

+ a surrogate key
+ the natural/business key
+ effective_start and effective_end timestamps (or end_date)
+ a current_flag

On change, you "close" the previous record and insert a new version. You must define how to detect changes and handle late arriving updates.

Implementation checklist:

1. Stage the incoming snapshot and deduplicate it on the natural key (keep the latest per key).
2. Compare tracked attributes against the current dimension row with a null-safe comparison.
3. Close changed rows: set `effective_end_at = now()` and `is_current = false`.
4. Insert changed rows plus brand-new keys with `effective_start_at = now()`, `effective_end_at = '9999-12-31'`, `is_current = true`.
5. Do it in one transaction or one `MERGE` so readers never see two current rows for a key.

Use a sentinel `9999-12-31` end date rather than `NULL` so range joins and `between` predicates behave the same on every engine, and use a null-safe comparison (`<=>` in Snowflake/MySQL, `IS DISTINCT FROM` in Postgres/Trino) so a `NULL -> value` change is detected.

The merge is two operations that must be atomic: close the changed current rows, then insert a new version for every changed or new key. If readers can see the intermediate state they may see zero or two current rows.

Idempotency rules:
+ Stage and deduplicate the incoming snapshot to one row per natural key before comparing.
+ Compare with a null-safe operator (`is distinct from`, `<=>`) so a null transition counts as a change.
+ Derive the new version start timestamp from a single run-level value, not from `now()` called per row, so reruns are consistent.
+ Guard the insert with `not exists (select 1 from dim where is_current)`, or a unique index that allows one current row per key.

Gotcha: a rerun with the same source snapshot must add no rows. If change detection compares against `now()` or the previous run time, a rerun mints spurious versions.

Worked example: staging contains `C123 enterprise`. The close step sets the open row to `valid_to = '2026-07-01'`, `is_current = false`. The insert step adds `customer_sk = 1002` with `valid_from = '2026-07-01'`, `valid_to = '9999-12-31'`, `is_current = true`, `version = 2`. Facts loaded after that resolve to 1002; facts with earlier event dates resolve to 1001.

Common mistake: matching facts on the natural key only. Without the date range or a pre-resolved surrogate key, a point-in-time report returns every version and multiplies revenue.

## What is a factless fact table?

A factless fact table records events that do not have numeric measures, for example "student attended class" or "user is eligible for campaign". The fact itself can be counted, and the table is useful for coverage, eligibility, and relationship tracking.

```sql
-- coverage: which products were on promotion in which store on which day
create table fact_promotion_coverage (
  store_sk bigint,
  product_sk bigint,
  promotion_sk bigint,
  date_sk int,
  primary key (store_sk, product_sk, promotion_sk, date_sk)
);

-- "products that sold" vs "products that were available":
select count(distinct p.product_sk) as products_on_promotion
from fact_promotion_coverage p
join dim_date d on d.date_sk = p.date_sk
where d.year_month = '2026-10';
```

Factless facts are also how you answer "what did *not* happen" - for example, stores that had no sales on a day require a coverage fact joined against the sales fact.

Two kinds of factless table:
+ Event factless: a row records that an event happened, such as a login or an attendance. Count rows, or count distinct entities.
+ Coverage factless: a row records that a relationship was possible, such as a product being on promotion or a customer being eligible. Join it against the activity fact to find what did not happen.

Mechanically, keep the same dimension keys as the activity fact and give it a primary key across them, for example `(store_sk, product_sk, promotion_sk, date_sk)`. Aggregate with `count(distinct ...)`, never `sum`.

Gotcha: adding a placeholder measure of `1` and summing it after a join double counts. A coverage row exists once per relationship; counting it after joining to a fact with many rows per relationship inflates the number.

Worked example: `fact_promotion_coverage` says which products were on promotion. "Which promoted products did not sell?" is an anti join against sales:

```sql
select distinct c.product_sk
from fact_promotion_coverage c
left join fact_order_line f
  on f.product_sk = c.product_sk
 and f.order_date_sk = c.date_sk
where f.product_sk is null;
```

Common mistake: storing the coverage fact at a coarser grain than the activity fact, so the anti join compares a month to a day and every product looks unsold on most days.

## What is a bridge table and when do you need it?

A bridge table resolves many-to-many relationships, for example products belonging to multiple categories or users having multiple roles. It can also store weights for allocation (for example revenue split). Without a bridge table, joins can explode row counts and break aggregations.

```sql
create table bridge_order_promotion (
  order_sk     bigint,
  promotion_sk bigint,
  allocation_weight numeric(5, 4),
  primary key (order_sk, promotion_sk)
);

-- weighted revenue by promotion, without double counting
select p.promotion_name,
       sum(f.revenue * b.allocation_weight) as attributed_revenue
from fact_order f
join bridge_order_promotion b on b.order_sk = f.order_sk
join dim_promotion p          on p.promotion_sk = b.promotion_sk
group by p.promotion_name;
```

Design points:

+ The weights for one order should sum to 1 (or the allocation is explicitly documented as unweighted).
+ Add a "primary" flag when one relationship should win for simple reports.
+ A bridge is also a conformed dimension helper: it lets two facts meet without a direct many-to-many join.

A bridge table is a fact-like table for a many-to-many relationship. Its grain is one row per relationship and its key is the pair of surrogate keys. When a measure must be split across the relationship it carries an `allocation_weight` whose rows sum to 1 per parent key.

How it works:
+ Reports that count relationships (`count(distinct parent_key)`) need no weight.
+ Reports that allocate a measure (`sum(measure * weight)`) need weights that are complete and sum to 1.
+ Weighted attribution reconciles to the parent total only if each parent's weights sum to exactly 1.

Gotcha: an unweighted join fans out. If one order has three promotions and you join the bridge to the order fact with no weight, revenue is counted three times. Either store weights or switch the metric to `count(distinct order_id)`.

Worked example: order `O1` has revenue 100 and two promotions. The bridge stores `(O1, P1, 0.5)` and `(O1, P2, 0.5)`. `sum(revenue * weight)` returns 50 + 50 = 100, so promotion totals sum back to the order total. Without weights the same query returns 200.

Common mistake: weights that sum to something other than 1 after a data error, for example 0.6 and 0.6. The promotion report no longer reconciles to total revenue, and the gap is invisible because no row is null.

## What are degenerate dimensions?

Degenerate dimensions are identifiers stored in the fact table without a separate dimension table, such as order_id or invoice_number. They are useful for drill-through and filtering while avoiding unnecessary dimension tables.

```sql
select order_id,            -- degenerate dimension
       sum(gross_amount) as order_total
from fact_order_line
group by order_id
having sum(gross_amount) > 10000;
```

Use degenerate dimensions when the identifier has no descriptive attributes of its own, when it is at the grain of the fact, and when its cardinality is too high to justify a dimension table. Keep it in the fact, index it if drill-through is frequent, and do not build a dimension that contains only the id.

Mechanics: a degenerate dimension is a dimension key with no dimension table. It sits in the fact because it is at the fact grain and has no descriptive attributes worth a join. `order_id` on an order-line fact and `invoice_number` on a billing fact are the classics.

Contrast with a junk dimension: many low-cardinality flags and codes with no natural home are collapsed into one small table so the fact does not grow a column per flag. A degenerate dimension keeps a high-cardinality identifier in the fact; a junk dimension moves many low-cardinality flags out of it.

Gotcha: a degenerate dimension is still part of the grain. If `order_id` appears on every line while you claim the fact is one row per order, the declared grain is wrong.

Worked example: `fact_order_line` is one row per line and carries `order_id` as a degenerate dimension. Drill-through groups by `order_id` to show the order total, and an operational lookup filters `where order_id = 'O1'`. No `dim_order` is created because the id has no attributes of its own.

Common mistake: building a dimension table that contains only the id. It adds a join and a load step, provides nothing, and hides the fact that the id is part of the grain.

## How do you prevent double counting in analytical models?

You prevent double counting by enforcing correct grain, using proper join keys (many-to-one), and avoiding many-to-many joins without a bridge table. You also define metric rules (sum vs distinct count vs last value) and test them with reconciliation queries.

The classic fan-out bug: joining a fact to a dimension that has more than one matching row per fact row multiplies the measure.

```sql
-- wrong: a customer with two addresses doubles every order amount
select sum(o.order_amount)
from fact_order o
join dim_customer_address a on a.customer_sk = o.customer_sk;

-- right: aggregate the dimension first, or filter to the primary address
select sum(o.order_amount)
from fact_order o
join (select customer_sk, min(address_sk) as address_sk
      from dim_customer_address
      group by customer_sk) a
  on a.customer_sk = o.customer_sk;
```

Other guards:

+ Prefer joining to the current dimension row (`is_current = true`) for point-in-time-insensitive reports, and to the versioned row for as-of reports.
+ Never join two facts on a common key unless one is aggregated to the same grain first.
+ For semi-additive measures such as balances, use `max` or `avg` over time, not `sum`.
+ Add a reconciliation test that compares the mart total to the source total; alert on drift above a threshold.

Fan-out happens when a join is not many-to-one. The three usual causes:
+ A dimension with duplicate keys, such as one row per address or per SCD2 version, joined without a filter.
+ A many-to-many relationship joined directly instead of through a bridge.
+ Two facts at different grains joined on a shared key, the chasm trap.

Detection: compare the fact row count before and after the join, and compare the measure total against the unjoined aggregate. Any change means the join multiplied rows.

Gotcha: `count(*)` and `sum()` fail differently. Fan-out inflates both, but `count(distinct key)` can hide it, so a wrong number sometimes looks right until someone asks for revenue.

```sql
-- detect fan-out before it reaches a dashboard
select
  (select count(*) from fact_order_line)                    as fact_rows,
  (select count(*) from fact_order_line f
     join dim_product p on p.product_sk = f.product_sk)     as joined_rows,
  (select count(*) from (
       select product_sk from dim_product
       group by product_sk having count(*) > 1) d)          as duplicate_keys;
```

Common mistake: assuming referential integrity implies uniqueness. A dimension can be complete and still hold several rows per key when SCD2 history is present. The join must target the current or as-of version explicitly.

## How do you validate a data model after changes?

Typical validation includes:

+ row count and key uniqueness checks at each layer
+ reconciliation of totals against source systems
+ sampling and drill-through checks on known entities
+ regression tests for key business metrics

This catches subtle issues like join duplication or missing late data.

```sql
-- uniqueness + not-null tests, in the style of dbt tests
select customer_id, count(*) as c
from dim_customer
where is_current
group by customer_id
having count(*) > 1;

-- exactly one open version per key
select customer_id, count(*) as open_versions
from dim_customer
where effective_end_at = date '9999-12-31'
group by customer_id
having count(*) <> 1;

-- referential integrity
select count(*) as orphan_facts
from fact_order_line f
left join dim_product p on p.product_sk = f.product_sk
where p.product_sk is null;
```

Make the tests part of the pipeline, not a manual step: fail the run when a test returns rows, and store the result next to the table so a reviewer can see what was checked.

Validation is a set of invariants, each expressed as a query that must return zero rows:
+ Grain uniqueness: the declared key is unique.
+ Dimension key uniqueness: one row per key per current version.
+ Referential integrity: no fact row points at a missing dimension key.
+ Reconciliation: mart totals equal source totals within a tolerance.
+ Coverage: no date or entity has an unexplained gap.

Run these as pipeline tests, not one-off checks. A test that returns rows should fail the load; a test that only warns is a test that will be ignored.

Gotcha: testing only the happy path. Test the known edges too: a customer with no segment, a fact with a null dimension key, a duplicate feed file.

In [ ]:
# stdlib model of the invariants a validation step should assert
def check_grain(rows, key):
    seen = set()
    for row in rows:
        k = tuple(row[col] for col in key)
        if k in seen:
            return False, k
        seen.add(k)
    return True, None

fact_lines = [
    {"order_id": "O1", "line_number": 1},
    {"order_id": "O1", "line_number": 2},
    {"order_id": "O2", "line_number": 1},
]
ok, duplicate = check_grain(fact_lines, ("order_id", "line_number"))
print("grain unique:", ok)
assert ok and duplicate is None

Worked example: cardinality expectations make fan-out visible before it reaches a report.

```sql
select count(*) as fact_rows,
       count(distinct order_id, line_number) as expected_rows
from fact_order_line;   -- these must be equal

select count(*) as dim_rows,
       count(distinct customer_sk) as unique_keys
from dim_customer;      -- dim_rows >= unique_keys because of SCD2 versions
```

Common mistake: validating only row counts. Two tables can have identical counts and still disagree, because a rewrite with wrong joins changes values without changing cardinality. Reconcile the measures too.

## How do you choose the grain for a fact table?

Start with the business process and write the grain as a sentence before naming columns. Good examples are "one row per order line", "one row per payment transaction", or "one row per account per day". Once the grain is fixed, every measure and dimension key must fit that level of detail.

Example:

+ `fact_order_line`: one row per order line item; measures include quantity, gross_amount, discount_amount
+ `fact_order`: one row per order; measures include order_total and shipping_amount
+ `fact_customer_day`: one row per customer per day; measures include daily_active_flag and session_count

The common mistake is mixing grains in one table, such as storing order-level shipping cost on every order line. That causes double counting unless you allocate the value or keep it in a separate order-grain fact.

```text
If the business asks for X and the answer needs Y detail,
model at Y and aggregate up - never the reverse.
```

When two candidate grains are valid, prefer the lowest granular one that still answers the main questions, because you can always aggregate up but you cannot split an aggregated row back into detail.

Grain selection is driven by the questions, the source event, and the cost of storage. Ask three things:
1. What is the lowest-level event the business records? That is usually the transactional grain: one row per line, per scan, per click.
2. Which measures must be summed, and across which dimensions? Additive measures need the event grain; semi-additive measures such as balances need a snapshot.
3. How much history and cardinality will the table hold? At billions of rows a coarser snapshot may be the deliberate choice.

Gotcha: the header and line pattern. Order-level values such as shipping and tax are correct only at order grain. Put them in `fact_order`, or allocate them to lines with a documented rule. Never store them in both.

Worked example: a requirement asks for daily active users, sessions per user per day, and monthly retention. The lowest grain that serves all three is one row per user per day: `fact_user_day` with `daily_active_flag` and `session_count`. Retention is a self-join across months at that grain. A coarser monthly table could show retention but not daily actives.

Common mistake: picking the grain of the report instead of the process. When the report changes from monthly to weekly, a monthly fact cannot be split back apart.

## What does an SCD Type 2 table look like in practice?

An SCD Type 2 dimension stores multiple versions of a business entity so historical facts can join to the correct version "as of" the event date. It usually has a surrogate key, the natural key, tracked attributes, effective dates, and a current flag.

Example:

```sql
create table dim_customer (
  customer_sk bigint,
  customer_id varchar(50),
  email varchar(255),
  customer_segment varchar(50),
  effective_start_at timestamp,
  effective_end_at timestamp,
  is_current boolean
);
```

If customer `C123` moves from `standard` to `enterprise`, close the old row by setting `effective_end_at`, then insert a new row with a new `customer_sk`. Facts can either store the resolved `customer_sk` at load time or join by `customer_id` and event timestamp.

The same change expressed as one idempotent `MERGE` on a Snowflake-style warehouse:

```sql
merge into dim_customer as tgt
using (
  select customer_id, email, customer_segment
  from stg_customer
  qualify row_number() over (partition by customer_id order by updated_at desc) = 1
) as src
on tgt.customer_id = src.customer_id
   and tgt.is_current
when matched and (
       tgt.email            is distinct from src.email
    or tgt.customer_segment is distinct from src.customer_segment
) then update set
       tgt.effective_end_at = current_timestamp(),
       tgt.is_current       = false;

-- insert the new versions (and brand-new customers)
insert into dim_customer (
  customer_sk, customer_id, email, customer_segment,
  effective_start_at, effective_end_at, is_current
)
select
  abs(hash(src.customer_id || current_timestamp()::varchar)),
  src.customer_id,
  src.email,
  src.customer_segment,
  current_timestamp(),
  '9999-12-31'::timestamp,
  true
from stg_customer src
left join dim_customer tgt
  on tgt.customer_id = src.customer_id
 and tgt.is_current
where tgt.customer_id is null
   or tgt.email            is distinct from src.email
   or tgt.customer_segment is distinct from src.customer_segment;
```

A PySpark implementation that does the same work as two DataFrame operations:

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

staged = (
    spark.table("stg_customer")
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
    ))
    .where(F.col("rn") == 1)
    .drop("rn")
)

current = spark.table("dim_customer").where(F.col("is_current"))

changed = (
    current.alias("c")
    .join(staged.alias("s"), "customer_id")
    .where(
        (F.col("c.email") != F.col("s.email"))
        | (F.col("c.customer_segment") != F.col("s.customer_segment"))
    )
    .select("c.customer_sk")
)

changed_ids = [row.customer_sk for row in changed.collect()]

Interview guidance: the important part is not the exact hash function. It is that the update (close old versions), the insert (open new versions), and the current-flag invariant happen atomically, and that rerunning the job for the same source snapshot produces the same dimension.

Point-in-time joins are the operational core of SCD2. A fact row resolves to the dimension version whose validity range contains the event timestamp: `event_ts >= valid_from and event_ts < valid_to`. The range is half-open so adjacent versions never both match.

At scale, two techniques keep it cheap:
+ Resolve the surrogate key at fact load time when facts arrive roughly in order, so reports join on a single key.
+ Keep the current row in a small table or view and the history in a table partitioned by `valid_from` year, so the common current-state query does not scan every version.

Gotcha: a range join on a wide history table can be expensive and cannot use an equi-join alone. Give the join an equality on the natural key first, then the range, and prune the history by partition.

```sql
-- as-of join: exactly one version per event, half-open range
select f.order_id,
       c.customer_segment as segment_at_order,
       f.gross_amount
from fact_order f
join dim_customer c
  on c.customer_id = f.customer_id
 and f.order_ts >= c.valid_from
 and f.order_ts <  c.valid_to;
```

Common mistake: using `between c.valid_from and c.valid_to` with inclusive end dates. Two adjacent versions then both match the boundary instant and the fact row is duplicated. Use `>= valid_from and < valid_to` everywhere, including in tests.

## How do you model many-to-many relationships without breaking metrics?

Use a bridge table instead of joining dimensions directly in a way that multiplies fact rows. The bridge stores one row per relationship, and sometimes an allocation weight when measures need to be split.

Example:

```text
fact_order
  order_id, revenue
```


If one order has two promotions and revenue is 100, a weighted bridge might store two rows with `allocation_weight = 0.5`. Then promotion revenue is calculated as `sum(revenue * allocation_weight)`, avoiding accidental double counting.

Alternative pattern when weights are unknown: report "orders influenced by promotion" as a distinct count on the bridge instead of a revenue sum.

```sql
select p.promotion_name,
       count(distinct b.order_id) as influenced_orders
from bridge_order_promotion b
join dim_promotion p on p.promotion_sk = b.promotion_sk
group by p.promotion_name;
```

Weighted attribution and impact counting are different metrics, and mixing them is the usual error.
+ Weighted attribution: `sum(measure * allocation_weight)`, which reconciles to the parent total.
+ Impact or influence: `count(distinct parent_key)`, which answers "how many orders were touched" and does not reconcile to revenue.
+ Unweighted spread: `sum(measure) / count(*)` or a ratio, valid only if the business accepts an even split.

Gotcha: a metric called "revenue influenced by promotion" but computed as `sum(measure)` over the bridge is wrong twice: it double counts and it is not a count.

Worked example: 100 orders, of which 30 have one promotion and 10 have two. Influenced orders is 40 by distinct count. Attributed revenue uses weights and sums to the promoted portion of total revenue. If an executive asks why attributed promotion revenue is smaller than total revenue, the answer is that only promoted orders carry a weight.

Common mistake: reusing the bridge for a metric with different semantics without renaming it, so two dashboards both display "promotion revenue" from different formulas.

## What is a metric definition layer and why does it matter?

A metric definition layer centralizes how business metrics are calculated so dashboards, notebooks, and data products do not invent competing formulas. It defines the numerator, denominator, filters, grain, time dimension, and allowed dimensions.

Example:

```yaml
metric: net_revenue
base_table: fact_order_line
expression: sum(gross_amount - discount_amount - refund_amount)
time_dimension: order_date
allowed_dimensions:
  - dim_customer.customer_segment
  - dim_product.category
```

This matters because most metric disputes are modeling problems, not SQL syntax problems. Clear metric contracts also make refactoring safer because tests can compare old and new outputs for key slices.

A more complete contract also states the grain, the default time window, the owner, and what is explicitly excluded:

```yaml
metric: net_revenue
owner: finance-analytics
base_table: gold.fact_order_line
grain: order_line
expression: sum(gross_amount - discount_amount - refund_amount)
filters:
  - order_status <> 'cancelled'
time_dimension: order_date
default_window: last_28_days
allowed_dimensions:
  - dim_customer.customer_segment
  - dim_customer.country_code
  - dim_product.category
currency: USD
```

Whether you implement it with dbt metrics, a semantic layer (Cube, Looker, AtScale), or a table of pre-aggregated metrics, the contract is what stops three dashboards from shipping three revenue numbers.

A metric contract must pin down additivity and grain, not just the expression:
+ Additive measure: safe to sum across all dimensions.
+ Semi-additive measure: safe across some dimensions; use last value or average over time.
+ Ratio metric: store the numerator and denominator, never the ratio.
+ Distinct-count metric: define exactly which key is counted.

Also record the default filters, the time dimension, the allowed dimensions, and the currency. Those are the parts that silently change between dashboards.

Gotcha: averaging an average. If each row is an average session length, `avg(avg_session_len)` weights every row equally regardless of how many sessions it summarizes. Store the numerator and denominator and compute the ratio.

Worked example: `average_order_value` is `sum(net_revenue) / count(distinct order_id)`. Both inputs live in the same fact at order-line grain, and the distinct count fixes the denominator. A precomputed `aov` per line would be meaningless.

Common mistake: defining the metric on one fact but letting a dashboard join another fact to add a filter, which changes the grain and therefore the number.

## What is a Data Vault model?

Data Vault 2.0 is an insert-only modeling approach built from three constructs:

+ **Hubs** - the unique business keys (customer, product, order).
+ **Links** - the relationships between hubs (order -> customer).
+ **Satellites** - the descriptive attributes and their history, attached to a hub or link.

It is designed for large, multi-source, auditable warehouses where ingestion must be parallel and repeatable. Every table is insert-only, loads are idempotent by hash key, and history is preserved without the SCD bookkeeping of a dimensional model.

```sql
-- hub: one row per business key ever seen
create table hub_customer (
  customer_hk  binary(16) primary key,   -- hash of the business key
  customer_id  varchar(50) not null,
  load_ts      timestamp,
  record_source varchar(50)
);

-- link: relationship between hubs
create table link_order_customer (
  link_hk      binary(16) primary key,
  order_hk     binary(16),
  customer_hk  binary(16),
  load_ts      timestamp,
  record_source varchar(50)
);

-- satellite: attributes with full history
create table sat_customer_details (
  customer_hk  binary(16),
  load_ts      timestamp,
  hash_diff    binary(16),   -- hash of the tracked attributes
  customer_name varchar(200),
  segment      varchar(50),
  load_end_ts  timestamp,
  primary key (customer_hk, load_ts)
);
```

In [ ]:
# hub load: keep the first occurrence of each business key
from pyspark.sql import functions as F

incoming = spark.table("stg_customer").select("customer_id", "load_ts", "record_source")
incoming = incoming.withColumn("customer_hk", F.md5(F.concat_ws("||", "customer_id")))

new_keys = incoming.join(
    spark.table("hub_customer").select("customer_hk"), "customer_hk", "left_anti"
)

new_keys.write.mode("append").insertInto("hub_customer")

Trade-offs:

+ **For**: parallel loads, full audit history, easy multi-source integration, resilient to source changes.
+ **Against**: many tables and joins, poor for direct BI consumption, needs a dimensional or wide mart layer on top. Data Vault is an integration pattern, not a reporting model.

Mechanics: hubs hold distinct business keys, links hold relationships, and satellites hold attributes with `load_ts`, `hash_diff`, and end-dating. Every insert is keyed by a hash of the business key, so a rerun inserts nothing new. A `hash_diff` over the tracked attributes lets the load detect change without comparing every column.

Raw Data Vault is expensive to query because one business entity is scattered across hubs, links, and satellites. Teams publish point-in-time (PIT) and bridge tables that pre-join the relevant satellites as of a snapshot date, then build dimensional marts on top.

Gotcha: doing BI directly on raw vault tables. It works for audit and lineage, not for dashboards, where every report re-derives the same joins.

Worked example: `hub_customer` has one row per `customer_id`. `sat_customer_details` gets a new row whenever `hash_diff` changes, with the previous row end-dated by the next `load_ts`. To answer "segment as of 2026-06-30", join the satellite with `load_ts <= '2026-06-30'` and take the latest row per hub key. That is the same as-of problem as SCD2, solved in the query rather than in the load.

Common mistake: putting the same attribute in two satellites, or in both the hub and a satellite. Duplicate ownership makes change detection ambiguous.

## How do you handle SCD Type 1 and Type 3?

**Type 1** overwrites the value and keeps no history. It is right for corrections and non-analytical attributes (typos, formatting, phone numbers) where the old value has no reporting value.

```sql
update dim_customer as tgt
set customer_name = src.customer_name,
    email         = src.email
from stg_customer src
where src.customer_id = tgt.customer_id
  and tgt.is_current;
```

**Type 3** keeps a limited history in extra columns, usually "current" and "previous". It is right when the business only ever compares the last change.

```sql
alter table dim_customer
  add column previous_segment varchar(50),
  add column segment_changed_at timestamp;

update dim_customer as tgt
set previous_segment   = tgt.customer_segment,
    customer_segment   = src.customer_segment,
    segment_changed_at = current_timestamp()
from stg_customer src
where src.customer_id = tgt.customer_id
  and tgt.is_current
  and tgt.customer_segment is distinct from src.customer_segment;
```

Guidance:

+ Type 1 is cheap but destroys as-of reporting; once history is gone it usually cannot be reconstructed.
+ Type 3 answers "what was it before?" only for one step; it cannot answer "what was it in March?".
+ If the business cannot state the retention requirement, default to Type 2 for analytical attributes and Type 1 for corrections.

Mechanics: Type 1 is an `update ... from` that overwrites the current row, and it is idempotent because the same input produces the same state. Type 3 adds a `previous_value` column and a changed-at timestamp; it answers "what was it before" but only one step back. Type 6 combines them: full Type 2 history plus a current-value column rewritten on every version of the entity, so one table serves both the current and historical views.

Gotcha: Type 6's current-value column must be updated on all versions of the entity in the same transaction, or historical rows show a stale value as current. That write amplification is the cost of the convenience.

Worked example: `customer_phone` is a correction attribute, so Type 1 overwrites with no history. `customer_segment` drives historical reporting, so Type 2 adds a version. A credit-rating migration needs only the previous rating, so Type 3 adds `previous_rating`. Choosing per attribute keeps history only where it earns its cost.

Common mistake: applying Type 2 to everything. Every typo fix then creates a version, the history doubles every few months, and as-of joins slow down for data nobody queries.

## How do you handle late-arriving dimensions and facts?

Late-arriving data is normal in batch warehouses: a fact for 2026-09-15 lands on 2026-10-04, or a new customer first appears in a sales file before the customer feed has it.

For **late-arriving facts**, keep the event date in the fact and load into the correct partition. The dimension join must resolve "as of" the event date, not "as of" the load date:

```sql
select f.order_id,
       coalesce(d.customer_segment, 'UNKNOWN') as customer_segment,
       f.gross_amount
from fact_order f
left join dim_customer d
  on d.customer_id = f.customer_id
 and f.order_ts >= d.effective_start_at
 and f.order_ts <  d.effective_end_at;
```

For **late-arriving dimensions**, create an inferred or placeholder row at fact-load time (a "UNKNOWN" member keyed by the natural key), then update that row when the real description arrives:

```sql
-- inferred member: facts can join immediately, attributes filled in later
insert into dim_customer (customer_sk, customer_id, customer_name, is_current,
                          effective_start_at, effective_end_at, is_inferred)
select abs(hash(customer_id)), customer_id, 'UNKNOWN', true,
       timestamp '1900-01-01', timestamp '9999-12-31', true
from (select distinct customer_id from stg_order) s
where not exists (
  select 1 from dim_customer d where d.customer_id = s.customer_id
);
```

Then run the SCD2 merge when the real customer record arrives; because the placeholder used the same natural key, facts already loaded will join to the corrected version. Document that inferred rows are temporary, and add a metric for how many rows are still inferred so the gap is visible.

Late data has three distinct problems:
+ Late fact, dimension already known: route the fact to the partition for its event date; never use the load date as the partition key.
+ Late fact, dimension unknown: create an inferred member so the fact has a valid key, then update it when the real record arrives.
+ Late dimension change whose effective date predates facts already loaded: an out-of-order version. You must adjust the validity range and may need to re-point affected fact rows, which is a restatement.

Control it with a restatement window: accept late data for the last N days and rebuild those partitions, while older corrections go through a documented backfill.

Gotcha: using load time as the event time. It shifts every metric when pipelines are delayed and destroys partition pruning by event date.

```sql
-- route a late fact to its event-date partition and resolve the as-of dimension
insert into fact_order (order_sk, customer_sk, order_date_sk, gross_amount)
select o.order_sk,
       c.customer_sk,
       cast(replace(o.order_date, '-', '') as int) as order_date_sk,
       o.gross_amount
from stg_order o
left join dim_customer c
  on c.customer_id = o.customer_id
 and o.order_ts >= c.valid_from
 and o.order_ts <  c.valid_to;
```

Common mistake: creating a separate "late data" table or partition. It doubles every downstream query and hides the gap instead of measuring it.

## What is a date dimension and a role-playing dimension?

A **date dimension** is one row per calendar day with pre-computed attributes: fiscal periods, week numbers, is_weekend, is_holiday, year-over-year comparison dates. It is the most common conformed dimension and it prevents every analyst from re-deriving fiscal calendars differently.

```sql
create table dim_date (
  date_sk       int primary key,      -- 20261004
  date_value    date not null,
  day_name      varchar(10),
  week_of_year  int,
  month_name    varchar(10),
  year_month    char(7),              -- 2026-10
  fiscal_quarter varchar(10),
  is_weekend    boolean,
  is_holiday    boolean,
  same_day_last_year date
);
```

A **role-playing dimension** is the same physical dimension used several times in one fact with different meanings. Classic examples: `order_date`, `ship_date`, and `delivery_date` all point at `dim_date`; `origin_airport` and `destination_airport` both point at `dim_airport`.

```sql
select dd.year_month as order_month,
       ds.year_month as ship_month,
       count(*)      as orders
from fact_order f
join dim_date dd on dd.date_sk = f.order_date_sk
join dim_date ds on ds.date_sk = f.ship_date_sk
group by 1, 2;
```

Implementation options: join the same table with aliases (simple, what BI tools expect), or publish views named `dim_order_date`, `dim_ship_date`. Prefer aliases and document the roles in the semantic layer; do not physically duplicate the date dimension.

Date dimension mechanics: use an integer `date_sk` in `YYYYMMDD` form so facts store four bytes instead of a timestamp, and precompute every calendar attribute the business argues about: fiscal period, week start, is_holiday, and the year-ago comparison date. Reserve rows for unknown (`0`) and not applicable (`19000101`) dates.

Role-playing mechanics: one physical `dim_date` joined several times with aliases. The optimizer sees the same table, so dimension scans can be reused, and the views per role add only naming.

Gotcha: building `dim_order_date`, `dim_ship_date`, and `dim_delivery_date` as separate physical tables. They drift the moment a holiday is added to only one of them.

Worked example: `fact_order` has `order_date_sk`, `ship_date_sk`, and `delivery_date_sk`. A stage-duration report joins `dim_date` three times with aliases and computes the difference between `date_value` columns. The same dimension answers fiscal-quarter reporting with no extra model.

Common mistake: storing raw timestamps in the fact and deriving calendar attributes in every query. Timezone and fiscal rules then live in dashboards instead of the model.

## Kimball vs Inmon: which modeling approach should you use?

| Aspect | Kimball (bottom-up) | Inmon (top-down) |
|---|---|---|
| Starting point | business processes and marts | enterprise-wide normalized model |
| Core structure | facts and dimensions | 3NF integration layer |
| Time to first value | fast (one mart) | slow (enterprise model first) |
| Consistency | conformed dimensions and bus matrix | enforced by the central model |
| Storage | some redundancy in dimensions | minimal redundancy |
| Best fit | BI-led, iterative delivery | large regulated enterprises with many sources |

A pragmatic hybrid is the most common answer in interviews:

1. Land raw data (bronze).
2. Normalize and conform in an integration layer (silver) - Inmon's discipline.
3. Publish dimensional marts (gold) with a bus matrix of conformed dimensions - Kimball's delivery model.

The **bus matrix** is the artifact that ties them together: rows are business processes, columns are conformed dimensions, and every cell marks whether a process captures that dimension. If a dashboard needs a slice that no cell provides, that is a modeling gap, not a SQL problem.

```text
                 dim_date  dim_customer  dim_product  dim_store
fact_order_line     X          X             X            X
fact_payment        X          X                          X
fact_inventory      X                        X            X
```

Mechanics of the two styles:
+ Inmon normalizes the integration layer to 3NF so each fact is stored once and the enterprise model is the single source of truth. Marts are built from it.
+ Kimball builds conformed dimensions and dimensional marts driven by business processes and the bus matrix.
+ Data Vault is a third option for large multi-source ingestion: insert-only hubs, links, and satellites, with dimensional marts published on top.

They are not exclusive at the layer level. A common production shape is 3NF or vault in silver and dimensional in gold.

Gotcha: presenting the choice as ideology. The interview answer is about constraints: regulatory audit and many sources favor a normalized core, while time to value and BI self-service favor dimensional marts.

Worked example: a bank with 40 source systems and a regulator uses a normalized core for lineage, Data Vault for parallel ingestion and audit, and a Kimball-style `fact_transaction` star for analysts. The bus matrix defines `dim_customer`, `dim_account`, and `dim_date` once and reuses them everywhere.

Common mistake: building dimensional marts straight from sources before the entities are conformed. Each mart invents its own customer definition and the drill-across numbers never agree.

## How do you model for performance and scale?

For this topic there is no tuning chapter in the database sense. Performance here comes from four modeling decisions: the grain you choose, how much you denormalize, how the model lines up with the physical layout, and how much history the SCD strategy keeps. Get these right and most query tuning becomes unnecessary.

## How do you choose the grain and pre-aggregate?


Pre-aggregation trades storage and refresh cost for query speed. A summary table is correct only if its grain is exactly the aggregation of the base fact and every consumer knows the filters and the dimensions it can no longer slice by.

Rules of thumb:
+ Pre-aggregate when the same heavy group-by runs constantly and its result is small.
+ Keep the base fact at event grain; summaries are derived, never the only copy.
+ Put the summary grain in its name, for example `fact_order_line_monthly_product`, and in the metric contract.
+ A summary at the wrong grain is worse than no summary, because it silently drops dimensions.

Gotcha: a summary that stores a non-additive measure such as a ratio cannot be re-aggregated across a dimension the summary already collapsed.

Worked example: a dashboard groups 500 million order lines by category and month. The pre-aggregate `fact_sales_monthly` holds 10 years x 12 months x 400 categories = 48,000 rows, so the dashboard scans kilobytes instead of gigabytes.

The cost of a wrong grain is concrete. If the summary stores `avg_order_value` per category-month, a yearly rollup averages monthly averages. Monthly average order value is not weighted by order count, so the yearly figure is wrong. Store `sum_revenue` and `order_count` in the summary and divide at query time.

Common mistake: adding a summary before the base grain is stable. When the grain changes, every summary and every dashboard built on it must be rebuilt.

## Denormalization vs joins: when does a star beat a snowflake on MPP?


On a columnar MPP engine the cost of a join is dominated by data movement, not by row lookups. Small dimensions are broadcast to every node and joined locally; large-to-large joins shuffle both sides across the network. That asymmetry decides the model.

+ A star keeps each dimension join fact-to-dimension, so the optimizer broadcasts a small dimension and the fact does not move.
+ A snowflaked chain can turn one join into several, and a mid-sized intermediate may no longer be broadcastable.
+ Columnar compression makes repeated dimension text cheap, so the storage argument for snowflaking is weak.
+ A wide dimension is scanned once and cached; a wide fact is scanned on every query.

Gotcha: denormalizing by copying attributes onto the fact removes joins but also removes the single point of truth. Do it as a materialized view over the star, not as a second hand-maintained table.

Worked example: `fact_order_line` with 500 million rows joins `dim_product` with 200 thousand rows, already flattened to category and department. The planner broadcasts `dim_product`, so no fact rows shuffle. Snowflaked, the planner first joins `dim_product` to `dim_category`, which is still small, so the numbers match, but the plan gains a stage and the optimizer has one more chance to choose a shuffle.

Common mistake: snowflaking a fact-to-fact relationship. Two facts at the same grain joined directly create a chasm trap; denormalization does not fix that, a bridge or a shared grain does.

## How do you align the model with the physical layout?


Logical grain and physical layout are separate decisions that must agree. The layout tools:
+ Partitioning splits the table into files by a column, usually an event date or tenant. A query that filters on the partition column prunes files; a query that does not scans everything.
+ Clustering or sort keys order rows within partitions so zone maps and file statistics can skip blocks. Put the low-cardinality filter columns first and the join keys after.
+ File size targets matter: many tiny files cost planning time, a few huge files limit parallelism. Compact regularly.

The model decides which columns exist; the layout decides which of them are cheap to filter on. An integer `order_date_sk` on the fact is what makes partition pruning possible; a raw timestamp string is not.

Gotcha: partitioning by a high-cardinality key such as `order_id` creates millions of tiny partitions. Partition by date or tenant and cluster by the high-cardinality keys.

```sql
-- Snowflake-style: cluster by the columns the model filters and joins on
create or replace table fact_order_line (
  order_line_sk bigint,
  order_id      varchar(50),
  customer_sk   bigint,
  product_sk    bigint,
  order_date_sk int,
  net_amount    numeric(14, 2)
)
cluster by (order_date_sk, customer_sk);

-- Lakehouse-style: partition files by event date, sort within files by the join key.
-- Iceberg or Delta: partition by order_date, then z-order or sort by customer_sk.
```

Worked example: `fact_order_line` is partitioned monthly by `order_date_sk` and clustered by `(customer_sk, product_sk)`. A query for one month and one customer reads one partition and skips most blocks inside it. The same query without the date filter must read every partition, because the customer is only ordered within a partition.

Common mistake: a model that joins on `customer_id` strings while the physical table is clustered on `customer_sk`. The join still works, but it cannot use the sort order. Align the semantic key in the model with the physical sort key.

## What is the SCD strategy at scale?


SCD2 cost grows with the change rate, not the entity count. A 12-million-customer dimension where 2 percent change per day grows by 240 thousand rows a day, about 88 million rows a year. Four things keep it manageable:
+ Partition the history by `valid_from` period so as-of queries prune old partitions.
+ Keep a small current view or current table for the common query.
+ Merge on the natural key with an equality predicate first, then the validity range, so the join is not a cross-product.
+ Archive closed versions older than the retention window when they are never queried.

Gotcha: rebuilding the whole dimension every run because the merge is easier to reason about. At 88 million rows a year a full rebuild eventually cannot finish inside the window, and the job becomes the constraint on the whole pipeline.

```sql
-- point-in-time join at scale: push the event range down so partitions prune
select f.order_id, c.customer_segment as segment_at_order
from fact_order f
join dim_customer c
  on c.customer_id = f.customer_id
 and f.order_ts >= c.valid_from
 and f.order_ts <  c.valid_to
where f.order_ts >= timestamp '2026-01-01';
```

Late-arriving changes are the hard case. If a change is reported with an effective date in the past, inserting it as a new version means closing the version that covered that date and reopening the remainder, which rewrites history. Prefer storing the source effective date and accepting a controlled out-of-order load over silently rewriting what was already reported.

Common mistake: using `is_current = true` as a shortcut for "as of today" in a report that is rerun historically. A rerun next month returns a different answer for the same period, which fails audit.

## Real world use cases for data modeling

### 1. Retail: order-line star with weighted promotion attribution

A retailer with ~40M order lines/month in Postgres needs margin reporting by category, store, and promotion. Order data is one row per line; promotions are many-to-many with orders and need revenue attribution.

```sql
create table fact_order_line (
  order_line_sk bigint primary key,
  order_id      varchar(50),
  customer_sk   bigint,
  product_sk    bigint,
  store_sk      bigint,
  order_date_sk int,
  quantity      int,
  gross_amount  numeric(14,2),
  discount_amount numeric(14,2),
  cost_amount   numeric(14,2)
);

-- attributed revenue: one order can have several promotions
select p.promotion_name,
       sum((f.gross_amount - f.discount_amount) * b.allocation_weight) as net_revenue
from fact_order_line f
join bridge_order_promotion b on b.order_id = f.order_id
join dim_promotion p          on p.promotion_sk = b.promotion_sk
join dim_date d               on d.date_sk = f.order_date_sk
where d.year_month = '2026-09'
group by p.promotion_name;
```

Why: the bridge keeps promotion revenue from double counting, the conformed date and product dimensions let margin roll up, and the star keeps the BI query to three joins.

### 2. Fintech: SCD2 customer dimension for regulator reporting

A payments company must report a customer's risk tier **as of** each transaction date, with a 7-year audit requirement. Customer attributes change a few thousand times a day across 12M customers.

```sql
select t.transaction_id,
       c.customer_segment   as segment_at_transaction,
       c.risk_tier          as risk_tier_at_transaction,
       t.amount
from fact_transaction t
join dim_customer c
  on c.customer_id = t.customer_id
 and t.transaction_ts >= c.effective_start_at
 and t.transaction_ts <  c.effective_end_at;
```

Why SCD2: reproducing a historical report requires the attribute values that were true then. Type 1 would silently rewrite history and fail the audit; the range join is the price of correctness.

### 3. SaaS: multi-tenant fact with tenant-scoped dimensions

A B2B SaaS platform serves 2,000 tenants and must never leak one tenant's data into another's dashboard. Every fact and dimension carries `tenant_sk`, and row-level security filters on it.

```sql
create table fact_usage_daily (
  tenant_sk bigint,
  account_sk bigint,
  usage_date_sk int,
  api_calls bigint,
  storage_gb numeric(12,2),
  primary key (tenant_sk, account_sk, usage_date_sk)
);

-- the tenant predicate is mandatory on every query
select d.usage_date, sum(f.api_calls)
from fact_usage_daily f
join dim_date d on d.date_sk = f.usage_date_sk
where f.tenant_sk = current_setting('app.tenant_sk')::bigint
group by d.usage_date;
```

Why: putting the tenant key at the lowest grain keeps the partition-pruning and row-level-security story simple, and it prevents cross-tenant joins from ever matching.

### 4. Marketing: factless coverage fact for campaign eligibility

A marketing team needs to answer "which customers were eligible for a campaign but did not convert?" Eligibility is not a numeric event, so it is modeled as a factless fact joined against conversions.

```sql
create table fact_campaign_eligibility (
  campaign_sk bigint,
  customer_sk bigint,
  date_sk     int
);

select count(distinct e.customer_sk) as eligible,
       count(distinct c.customer_sk) as converted
from fact_campaign_eligibility e
left join fact_conversion c
  on c.customer_sk = e.customer_sk
 and c.campaign_sk = e.campaign_sk
where e.campaign_sk = 42;
```

Why factless: there is no measure to sum, only the existence of a relationship. A factless fact is the cleanest way to store coverage and answer "what did not happen".

### 5. Logistics: accumulating snapshot for shipment milestones

A logistics company tracks each shipment from booking to delivery and reports stage durations daily. The process has a fixed set of milestones, so an accumulating snapshot updates one row per shipment as each milestone is reached.

```sql
update fact_shipment
set picked_up_date_sk  = 20261004,
    delivered_date_sk  = case when delivered_at is not null then 20261005 else delivered_date_sk end,
    days_in_transit    = datediff(day, booked_at, delivered_at)
where shipment_id = 'SHP-9001';
```

Why: accumulating snapshots make process-duration and pipeline-lag questions cheap (`avg(days_in_transit)`) without scanning an event stream, at the cost of in-place updates that must be idempotent.

## Data modeling exercises

These exercises use only the Python standard library and model the modeling decisions with small, runnable calculations. Each one names the task, gives a hint, states the expected result, and then asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The exercises model the arithmetic behind grain choice, SCD2 validity ranges, join fan-out, star versus snowflake, history growth, point-in-time joins, bridge sizing, and mixed-grain detection. Treat the asserted numbers as a check on the reasoning rather than a prediction of a real table; the value is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Pick a grain from a requirement

**Task.** Given three reporting requirements and a granularity ranking, find the single finest grain that can answer all of them.

**Hint.** A requirement is answerable from any grain at or below its required level, because you can aggregate up but never split a row apart. Pick the finest valid candidate.

**Expected result.** The requirements need level 1 or finer, so the chosen grain is `order_line`.

In [ ]:
# lower number = finer grain; a requirement can be answered from that level or finer
GRANULARITY = {"order_month": 3, "order": 2, "order_line": 1}
REQUIREMENTS = {
    "revenue by product by month": 2,   # month rollup needs an order-level date at least
    "average order value": 2,           # needs a countable order key
    "units per customer per day": 1,    # needs day-level line detail
}

finest_required = max(REQUIREMENTS.values())
valid = sorted(g for g, level in GRANULARITY.items() if level <= finest_required)
chosen = min(valid, key=lambda g: GRANULARITY[g])

print("requirements need level <=", finest_required)
print("candidate grains:", valid)
print("chosen grain:", chosen)
assert chosen == "order_line"
assert GRANULARITY[chosen] == 1

### Try it 2. Compute SCD2 rows from a change history

**Task.** Turn a chronologically ordered change history for one customer into SCD2 rows with `valid_from`, `valid_to`, `is_current`, and `version`.

**Hint.** Each version ends where the next begins, so `valid_to` equals the next change date and the range is half-open. The last version ends at the sentinel `9999-12-31`.

**Expected result.** Three versions, contiguous ranges, and exactly one current row with the sentinel end date.

In [ ]:
CHANGES = [
    ("2026-01-01", "standard"),
    ("2026-07-01", "enterprise"),
    ("2026-09-15", "enterprise_plus"),
]
SENTINEL = "9999-12-31"

rows = []
for i, (start, segment) in enumerate(CHANGES):
    end = CHANGES[i + 1][0] if i + 1 < len(CHANGES) else SENTINEL
    rows.append({
        "segment": segment,
        "valid_from": start,
        "valid_to": end,
        "is_current": i == len(CHANGES) - 1,
        "version": i + 1,
    })

for row in rows:
    print(row)

assert len(rows) == 3
assert rows[0]["valid_to"] == rows[1]["valid_from"]   # half-open, no overlap
assert sum(row["is_current"] for row in rows) == 1
assert rows[-1]["valid_to"] == SENTINEL

### Try it 3. Detect fan-out from join cardinalities

**Task.** Given row counts per fact key and the number of matching dimension rows per key, compute the joined row count and the inflated measure.

**Hint.** Joined rows are `fact_rows_per_key x dim_rows_per_key`, summed over keys. The measure scales by the same ratio when the join fans out.

**Expected result.** 1,800 joined rows, a 1.8x multiplier, and an inflated measure of 90,000 from a true 50,000.

In [ ]:
fact = {"total_rows": 1000, "total_amount": 50000}
fact_rows_per_key = {1: 400, 2: 400, 3: 200}
dim_rows_per_key = {1: 2, 2: 1, 3: 3}   # key -> matching dimension rows

joined_rows = sum(
    fact_rows_per_key[k] * dim_rows_per_key[k] for k in fact_rows_per_key
)
ratio = joined_rows / fact["total_rows"]
inflated = fact["total_amount"] * ratio

print(f"joined rows = {joined_rows} ({ratio:.2f}x fact)")
print(f"inflated measure = {inflated:.0f} from {fact['total_amount']}")
assert joined_rows == 1800
assert ratio == 1.8
assert inflated == 90000

### Try it 4. Choose star or snowflake from query patterns

**Task.** Given how many queries need one, two, or three dimension hops, compute the weighted average hops for a star and a snowflake, and pick the shape.

**Hint.** Multiply each pattern's query count by the hops it needs, sum, and divide by total queries. The shape with fewer weighted hops wins on a columnar MPP engine.

**Expected result.** Star average 1.0 hop versus snowflake 2.4 hops, so the recommendation is a star.

In [ ]:
QUERY_PATTERNS = {"single_dimension": 700, "two_hops": 200, "three_hops": 100}
STAR_HOPS = {"single_dimension": 1, "two_hops": 1, "three_hops": 1}
SNOWFLAKE_HOPS = {"single_dimension": 2, "two_hops": 3, "three_hops": 4}

total_queries = sum(QUERY_PATTERNS.values())

def weighted_hops(hops):
    return sum(QUERY_PATTERNS[p] * hops[p] for p in QUERY_PATTERNS) / total_queries

star = weighted_hops(STAR_HOPS)
snowflake = weighted_hops(SNOWFLAKE_HOPS)
recommendation = "star" if star < snowflake else "snowflake"

print(f"star weighted hops     = {star:.2f}")
print(f"snowflake weighted hops = {snowflake:.2f}")
print("recommendation:", recommendation)
assert round(star, 2) == 1.0
assert round(snowflake, 2) == 2.4
assert recommendation == "star"


### Try it 5. Compute history growth from change rate and retention

**Task.** For 12 million customers with 2 percent changing per day and a 7-year retention requirement, compute new SCD2 rows per day, per year, and over the full retention window.

**Hint.** `change_rows_day = entities x rate`; a 365-day year gives the annual figure; multiply by retention years for the total.

**Expected result.** 240,000 rows per day, 87,600,000 per year, and 613,200,000 over 7 years.

In [ ]:
entities = 12_000_000
change_rate = 0.02
retention_years = 7

rows_per_day = entities * change_rate
rows_per_year = rows_per_day * 365
rows_retained = rows_per_year * retention_years

print(f"new versions per day  = {rows_per_day:,.0f}")
print(f"new versions per year = {rows_per_year:,.0f}")
print(f"rows retained (7y)    = {rows_retained:,.0f}")
assert rows_per_day == 240_000
assert rows_per_year == 87_600_000
assert rows_retained == 613_200_000

### Try it 6. Plan a point-in-time join

**Task.** Resolve three events against two SCD2 versions using half-open validity ranges, and check that each event matches exactly one version.

**Hint.** The predicate is `valid_from <= event_ts < valid_to`. An event exactly on a boundary must pick the new version, not the old one.

**Expected result.** Each event matches exactly one version: 2026-06-30 to version 1001, and both 2026-07-01 and 2026-10-04 to version 1002.

In [ ]:
VERSIONS = [
    {"sk": 1001, "valid_from": "2026-01-01", "valid_to": "2026-07-01", "segment": "standard"},
    {"sk": 1002, "valid_from": "2026-07-01", "valid_to": "9999-12-31", "segment": "enterprise"},
]
EVENTS = [("2026-06-30", 1001), ("2026-07-01", 1002), ("2026-10-04", 1002)]

resolved = []
for event_ts, expected_sk in EVENTS:
    matches = [
        v for v in VERSIONS if v["valid_from"] <= event_ts < v["valid_to"]
    ]
    print(event_ts, [(mv["sk"], mv["segment"]) for mv in matches])
    assert len(matches) == 1
    assert matches[0]["sk"] == expected_sk
    resolved.append(matches[0]["sk"])

assert resolved == [1001, 1002, 1002]

### Try it 7. Size a bridge table from relationship cardinality

**Task.** For 8 million orders per month, where 15 percent carry two promotions and 5 percent carry three, compute the bridge rows per month and per year at order grain. Compare with the 40 million order lines per month.

**Hint.** Weighted relationships per order is `0.15 x 2 + 0.05 x 3 + 0.80 x 0`. Multiply by orders per month, then by 12.

**Expected result.** 3,600,000 bridge rows per month and 43,200,000 per year, far smaller than the 40 million line-grain alternative.

In [ ]:
orders_per_month = 8_000_000
order_lines_per_month = 40_000_000
PROMOTIONS_PER_ORDER = {2: 0.15, 3: 0.05}   # count -> share of orders

relationships_per_order = sum(count * share for count, share in PROMOTIONS_PER_ORDER.items())
bridge_rows_month = orders_per_month * relationships_per_order
bridge_rows_year = bridge_rows_month * 12

print(f"relationships per order = {relationships_per_order:.2f}")
print(f"bridge rows per month   = {bridge_rows_month:,.0f}")
print(f"bridge rows per year    = {bridge_rows_year:,.0f}")
print(f"line-grain alternative  = {order_lines_per_month:,.0f} per month")
assert round(relationships_per_order, 2) == 0.45
assert bridge_rows_month == 3_600_000
assert bridge_rows_year == 43_200_000

### Try it 8. Detect a mixed grain in one fact table

**Task.** Shipping cost is stored on every order line, but it is an order-level value. Compare the true shipping total with the total you get by summing at line grain.

**Hint.** At line grain the order-level value repeats once per line, so the naive sum is the true total multiplied by the number of lines per order.

**Expected result.** True total 35, naive total 115, so the table mixes order grain into a line-grain fact.

In [ ]:
order_lines = {"O1": 3, "O2": 1, "O3": 4}
shipping_per_order = {"O1": 10, "O2": 5, "O3": 20}

true_total = sum(shipping_per_order.values())
naive_total = sum(shipping_per_order[o] * n for o, n in order_lines.items())
overcount = naive_total / true_total

print("true shipping total:       ", true_total)
print("summed at line grain:      ", naive_total)
print(f"over-count factor:         {overcount:.2f}x")
assert true_total == 35
assert naive_total == 115
assert round(overcount, 2) == 3.29

Common mistake across all eight exercises: computing the right number from the wrong grain. Each exercise checks a formula, but the modeling judgment is choosing which formula the question actually requires.